# Implement PCA using both the Sample Covariance Matrix (Eigendecomposition) and SVD, then extend it to Kernel PCA (RBF kernel) for non-linear dimensionality reduction.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

class pca:
    def __init__(self, no_of_components=15):
        self.no_of_components= no_of_components
        self.variance_=None
        self.PCs_ = None
        self.mean_ = None

    def fit(self, X):
        N, D = X.shape # no. of datapoints and features
        self.mean_ = np.mean(X, axis=0)

        X_centered = X - self.mean_[None, :]

        U, S, VT= np.linalg.svd(X_centered, full_matrices=False)

        self.PCs_ = VT[:self.no_of_components]

        self.variance_ = ((S**2) / (N - 1)) [:self.no_of_components]
        

        return self

    def transform(self, X):
        X_centered = X - self.mean_[None, :]
        return X_centered@(self.PCs_.T)


class kernel_pca:
    def __init__(self, no_of_components = 15, gamma = 1.0):
        self.no_of_components = no_of_components
        self.gamma = gamma
        self.K_centered_ = None
        self.alphas_ = None
        self.lambdas_ = None
        self.X_fit_ = None
        self.K_fit_rows_mean_ = None
        self.K_fit_all_mean_ = None


    def rbf_kernel(self, X1, X2):

        X1_squar = np.sum(X1**2, axis = 1)[:, None]
        X2_squar = np.sum(X2**2, axis = 1)[None, :]

        dist = X1_squar + X2_squar - 2*(X1@(X2.T))
        return np.exp(-self.gamma*dist)

    def fit(self, X):
        self.X_fit_ = X
        N, D = X.shape
        kernel = self.rbf_kernel(X, X)
        one_mat = np.ones((N,N))/N

        self.K_fit_rows_mean_ = np.mean(kernel, axis=0, keepdims=True)  # (1, N)
        self.K_fit_all_mean_ = np.mean(kernel)

        self.K_centered_ =  kernel - one_mat@kernel - kernel@one_mat + one_mat@(kernel@one_mat)     # Double Centering 

        eigvals, eigvecs = np.linalg.eigh(self.K_centered_)
        idx= np.argsort(eigvals)[::-1]
        eigvals = eigvals[idx]
        eigvecs = eigvecs[:, idx]

        positive_idx = eigvals > 1e-10
        self.lambdas_ = eigvals[positive_idx][:self.no_of_components]
        self.alphas_ = eigvecs[:, positive_idx][:, :self.no_of_components]

        self.alphas_ = self.alphas_ / np.sqrt(self.lambdas_)
        return self

    def transform(self, X_test):
        K_test = self.rbf_kernel(X_test, self.X_fit_)  # (M, N)
        
        # Center test kernel using training kernel statistics
        K_test_rows_mean = np.mean(K_test, axis=1, keepdims=True)  # (M, 1)
        K_test_centered = K_test - self.K_fit_rows_mean_ - K_test_rows_mean + self.K_fit_all_mean_

        # Project centered kernel onto normalized eigenvectors
        return K_test_centered @ self.alphas_






rng = np.random.default_rng(seed=42)
X = rng.normal(loc = 0.0, scale = 1.0, size = (500, 50))
X_test = rng.normal(loc = 0.0, scale = 1.0, size = (100, 50))

pca_model = pca(no_of_components=15).fit(X)
Z_test_pca = pca_model.transform(X_test)
print("Linear PCA projected test shape:", Z_test_pca.shape)

# Kernel PCA
kpca_model = kernel_pca(no_of_components=15, gamma=0.01).fit(X)
Z_test_kpca = kpca_model.transform(X_test)
print("Kernel PCA projected test shape:", Z_test_kpca.shape)